# CatBoost Regression

training catboost regression in blocks using baseline features + traficom features


## 1. Imports

In [1]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load data

In [2]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [3]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

for dataset_df in [train_df, validation_df]:
    for date_column in ["first_seen_date", "last_seen_date", "scrape_date"]:
        if date_column in dataset_df.columns:
            dataset_df[date_column] = pd.to_datetime(dataset_df[date_column], errors="coerce")

reference_first_seen_date = min(
    df["first_seen_date"].min()
    for df in [train_df, validation_df]
    if "first_seen_date" in df.columns
)

for dataset_df in [train_df, validation_df]:
    dataset_df["first_seen_day_offset"] = (
        dataset_df["first_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["last_seen_day_offset"] = (
        dataset_df["last_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["listing_midpoint_day_offset"] = (
        dataset_df["first_seen_day_offset"]
        + dataset_df["last_seen_day_offset"]
    ) / 2


## 3. Quick data checks

In [4]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 82)
Validation shape: (1689, 82)


In [5]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 82 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   product_id                      7954 non-null   int64         
 1   part_name                       7954 non-null   object        
 2   price                           7954 non-null   float64       
 3   quality_grade                   7954 non-null   object        
 4   oem_number                      7954 non-null   object        
 5   mileage                         7954 non-null   float64       
 6   brand                           7954 non-null   object        
 7   model                           7954 non-null   object        
 8   category                        7954 non-null   object        
 9   subcategory                     7954 non-null   object        
 10  scrape_date                     7954 non-null   datetime64[ns

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [6]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column in train_df.columns

]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_day_offset": "Numeric position of listing start within the crawl window for CatBoost.",
    "last_seen_day_offset": "Numeric position of listing end within the crawl window for CatBoost.",
    "listing_midpoint_day_offset": "Numeric midpoint of listing visibility window for CatBoost.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

catboost_specific_exclusion_reasons = {
    "first_seen_date": "Raw date string replaced with numeric day-offset feature for CatBoost.",
    "last_seen_date": "Raw date string replaced with numeric day-offset feature for CatBoost.",
}

recommended_excluded_features = set(recommended_exclusion_reasons)
catboost_excluded_features = recommended_excluded_features | set(catboost_specific_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in catboost_excluded_features
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
    + [
        {
            "feature": column,
            "status": "xgboost_specific_exclusion",
            "reason": catboost_specific_exclusion_reasons[column],
        }
        for column in sorted(catboost_specific_exclusion_reasons)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nCatBoost-specific exclusions:")
print(sorted(catboost_specific_exclusion_reasons))

print("\nNumber of recommended model features:", len(recommended_model_features))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
3,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
4,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
5,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
6,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
7,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...
8,first_seen_date,xgboost_specific_exclusion,Raw date string replaced with numeric day-offs...
9,last_seen_date,xgboost_specific_exclusion,Raw date string replaced with numeric day-offs...


## 5. Select baseline feature set

In [7]:
features_baseline_only = list(dict.fromkeys(
    [
        feature for feature in baseline_features
        if feature not in catboost_excluded_features
    ]
    + [
        "first_seen_day_offset",
        "last_seen_day_offset",
        "listing_midpoint_day_offset",
    ]
))

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 16


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset']

## 6. Build X and y

In [8]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [9]:
y_train_log = np.log(y_train)
y_validation_log = np.log(y_validation)

y_train_log.head()


0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [10]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset']

## 8. Detect column types

In [11]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [12]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,8,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,8,"[part_name, quality_grade, oem_number, brand, ..."


## 9. Tune Native CatBoost


In [13]:
catboost_candidate_params = {
    "ordered_balanced_rmse": {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": 5000,
        "learning_rate": 0.02,
        "depth": 6,
        "l2_leaf_reg": 12,
        "random_strength": 1.0,
        "bagging_temperature": 0.5,
        "border_count": 254,
        "one_hot_max_size": 10,
        "rsm": 0.8,
        "boosting_type": "Ordered",
    },
    "ordered_conservative_rmse": {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": 6500,
        "learning_rate": 0.015,
        "depth": 5,
        "l2_leaf_reg": 15,
        "random_strength": 1.5,
        "bagging_temperature": 1.0,
        "border_count": 254,
        "one_hot_max_size": 10,
        "rsm": 0.8,
        "boosting_type": "Ordered",
    },
    "ordered_huber_robust": {
        "loss_function": "Huber:delta=0.12",
        "eval_metric": "RMSE",
        "iterations": 5000,
        "learning_rate": 0.02,
        "depth": 6,
        "l2_leaf_reg": 14,
        "random_strength": 1.2,
        "bagging_temperature": 0.75,
        "border_count": 254,
        "one_hot_max_size": 10,
        "rsm": 0.8,
        "boosting_type": "Ordered",
    },
    "plain_depth6_reference": {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": 4000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 8,
        "random_strength": 1.0,
        "bagging_temperature": 0.5,
        "border_count": 254,
        "one_hot_max_size": 10,
        "rsm": 0.8,
        "boosting_type": "Plain",
    },
}


def prepare_catboost_frame(df):
    prepared_df = df.copy()

    datetime_columns = prepared_df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()
    if len(datetime_columns) > 0:
        prepared_df = prepared_df.drop(columns=datetime_columns)

    object_columns = prepared_df.select_dtypes(include=["object", "category"]).columns.tolist()
    for column in object_columns:
        prepared_df[column] = prepared_df[column].astype("string").fillna("__missing__")

    bool_columns = prepared_df.select_dtypes(include=["bool"]).columns.tolist()
    for column in bool_columns:
        prepared_df[column] = prepared_df[column].astype(int)

    return prepared_df


def get_catboost_cat_features(prepared_df):
    return prepared_df.select_dtypes(include=["string"]).columns.tolist()


def make_catboost_regressor(params=None):
    model_params = {
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
    }
    if params is not None:
        model_params.update(params)

    return CatBoostRegressor(**model_params)


def fit_catboost_model(X_train_current, y_train_log_current, X_validation_current, y_validation_log_current, params):
    X_train_prepared_current = prepare_catboost_frame(X_train_current)
    X_validation_prepared_current = prepare_catboost_frame(X_validation_current)
    cat_features_current = get_catboost_cat_features(X_train_prepared_current)

    model_current = make_catboost_regressor(params)
    model_current.fit(
        X_train_prepared_current,
        y_train_log_current,
        cat_features=cat_features_current,
        eval_set=(X_validation_prepared_current, y_validation_log_current),
        use_best_model=True,
        early_stopping_rounds=200,
        verbose=False,
    )

    return model_current, X_train_prepared_current, X_validation_prepared_current, cat_features_current


In [14]:
tuning_features = recommended_model_features

X_train_tuning = train_df[tuning_features].copy()
X_validation_tuning = validation_df[tuning_features].copy()

tuning_results = []
for config_name, config_params in catboost_candidate_params.items():
    (
        model_current,
        X_train_tuning_prepared,
        X_validation_tuning_prepared,
        cat_features_tuning,
    ) = fit_catboost_model(
        X_train_tuning,
        y_train_log,
        X_validation_tuning,
        y_validation_log,
        config_params,
    )

    validation_pred_log_current = model_current.predict(X_validation_tuning_prepared)
    validation_pred_current = np.exp(validation_pred_log_current)

    tuning_results.append({
        "config": config_name,
        "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
        "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
        "validation_R2": r2_score(y_validation, validation_pred_current),
        "best_iteration": model_current.get_best_iteration(),
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_RMSE", "validation_MAE"]
).reset_index(drop=True)

selected_catboost_config_name = tuning_results_df.iloc[0]["config"]
selected_catboost_params = catboost_candidate_params[selected_catboost_config_name]

print("Selected CatBoost config:", selected_catboost_config_name)
print(selected_catboost_params)

display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)


Selected CatBoost config: plain_depth6_reference
{'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'iterations': 4000, 'learning_rate': 0.03, 'depth': 6, 'l2_leaf_reg': 8, 'random_strength': 1.0, 'bagging_temperature': 0.5, 'border_count': 254, 'one_hot_max_size': 10, 'rsm': 0.8, 'boosting_type': 'Plain'}


,config,validation_MAE,validation_RMSE,validation_R2,best_iteration
0,plain_depth6_reference,22.6975,150.7555,0.9293,3704
1,ordered_balanced_rmse,27.9343,162.6845,0.9177,4989
2,ordered_conservative_rmse,30.2841,163.8785,0.9165,6496
3,ordered_huber_robust,30.0351,199.5283,0.8762,4996


## 10. Train tuned baseline CatBoost


In [15]:
baseline_model, X_train_prepared, X_validation_prepared, baseline_cat_features = fit_catboost_model(
    X_train,
    y_train_log,
    X_validation,
    y_validation_log,
    selected_catboost_params,
)


## 11. Predict and convert back to euro scale

In [16]:
validation_pred_log = baseline_model.predict(X_validation_prepared)


In [17]:
validation_pred = np.exp(validation_pred_log)

## 12. Preview Baseline CatBoost Predictions


In [18]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,246.185916
1,473.6,370.182549
2,142.1,137.536062
3,82.9,58.235266
4,177.6,134.047286


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [19]:
features_baseline_plus_traficom = list(dict.fromkeys(
    features_baseline_only + traficom_features
))

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 29


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [20]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [21]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [22]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'last_seen_day_offset', 'listing_midpoint_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Prepare and train baseline + Traficom CatBoost


In [23]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()


In [24]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

(
    catboost_traficom,
    X_train_traficom_prepared,
    X_validation_traficom_prepared,
    cat_features_traficom,
) = fit_catboost_model(
    X_train_traficom,
    y_train_log,
    X_validation_traficom,
    y_validation_log,
    selected_catboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'last_seen_day_offset', 'listing_midpoint_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['part_name', 'quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 17. Predict baseline + Traficom CatBoost


In [25]:
validation_pred_log_traficom = catboost_traficom.predict(X_validation_traficom_prepared)


## 18. Predict on validation and convert back to euro scale

In [26]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)


In [27]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Preview Baseline + Traficom CatBoost Predictions


In [28]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,233.671875
1,473.6,368.331611
2,142.1,140.917977
3,82.9,62.297359
4,177.6,124.433707


In [29]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,233.868402
std,567.153248,462.540670
min,5.900000,8.712683
25%,59.000000,53.246378
50%,100.600000,92.850324
75%,236.000000,244.873886
max,4284.000000,3831.820688


## 20. All Recommended Features

This section trains the CatBoost model on every recommended feature while keeping the same validation structure used across the other notebooks.


In [30]:
features_all = recommended_model_features

print("Number of recommended model features:", len(features_all))
features_all

Number of recommended model features: 73


['part_name',
 'quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_

## 21. Build X and y for all recommended features

In [31]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [32]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [33]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Prepare and train recommended-feature CatBoost


In [34]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()


In [35]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

(
    catboost_all,
    X_train_all_prepared,
    X_validation_all_prepared,
    cat_features_all,
) = fit_catboost_model(
    X_train_all,
    y_train_log,
    X_validation_all,
    y_validation_log,
    selected_catboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 24. Predict recommended-feature CatBoost


In [36]:
validation_pred_log_all = catboost_all.predict(X_validation_all_prepared)


## 25. Predict on validation and convert back to euro scale

In [37]:
validation_pred_all = np.exp(validation_pred_log_all)


In [38]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [39]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,186.250364
1,473.6,478.779777
2,142.1,142.941887
3,82.9,78.636941
4,177.6,173.042093


In [40]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,251.787430
std,567.153248,548.638790
min,5.900000,6.636946
25%,59.000000,53.808823
50%,100.600000,101.408277
75%,236.000000,233.228721
max,4284.000000,4462.346971


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing CatBoost model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [41]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",186.250364,8.650364,74.828800
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",478.779777,5.179777,26.830095
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",142.941887,0.841887,0.708773
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",78.636941,4.263059,18.173671
4,177.6,airbag,all,"Side airbags - , e-(Right)",173.042093,4.557907,20.774517


In [42]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [43]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,6.95,3.41,10.42,75.7%,92.8%,100.0%
1,brakes,214,9.77,3.87,28.30,81.3%,94.9%,96.7%
2,vehicle exterior / suspension,294,10.17,2.28,36.14,86.1%,92.5%,95.9%
3,electric / transmitter / databox / sensor,404,12.79,2.95,45.53,76.2%,88.6%,95.3%
4,airbag,106,18.02,9.30,32.05,51.9%,70.8%,95.3%
5,gear box / drive axle / middle axle,241,25.04,5.04,106.40,71.4%,85.1%,92.5%
6,engine,249,75.84,3.53,370.26,70.3%,81.5%,91.2%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,75.84,3.53,370.26,70.3%,81.5%,91.2%
5,gear box / drive axle / middle axle,241,25.04,5.04,106.40,71.4%,85.1%,92.5%
4,airbag,106,18.02,9.30,32.05,51.9%,70.8%,95.3%
3,electric / transmitter / databox / sensor,404,12.79,2.95,45.53,76.2%,88.6%,95.3%
2,vehicle exterior / suspension,294,10.17,2.28,36.14,86.1%,92.5%,95.9%
1,brakes,214,9.77,3.87,28.30,81.3%,94.9%,96.7%
0,fuel,181,6.95,3.41,10.42,75.7%,92.8%,100.0%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,303.33,25.22,639.89,23.8%,42.9%,57.1%
22,passenger airbag,25,27.94,14.03,49.41,24.0%,76.0%,88.0%
21,adaptiv farthållare sensor,24,24.96,25.44,27.96,0.0%,50.0%,100.0%
20,contact roll airbag,20,15.00,10.74,16.53,35.0%,65.0%,100.0%
19,fuel filter / holder,24,12.75,13.37,14.21,25.0%,75.0%,100.0%
18,left front,39,10.68,3.72,31.89,79.5%,94.9%,94.9%
17,airbag control unit,20,9.33,1.72,16.84,60.0%,90.0%,95.0%
16,fuse box / electricity central,27,8.92,5.97,11.08,70.4%,100.0%,100.0%
15,deliverer,23,8.19,8.60,10.18,52.2%,100.0%,100.0%
14,all,153,6.51,3.27,13.77,86.3%,92.2%,99.3%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,3.08,2.21,4.28,90.0%,100.0%,100.0%
1,Wheel bearing spindle shaft -(Left rear),21,3.41,1.81,8.75,95.2%,95.2%,100.0%
2,Trailing link rear -(Left),20,4.71,2.93,6.28,95.0%,100.0%,100.0%
3,ABS Hydraulic pump -,24,5.39,3.48,7.14,75.0%,100.0%,100.0%
4,Wheel bearing spindle shaft -(Right rear),20,5.50,0.88,12.90,85.0%,85.0%,100.0%
5,Drive shaft -(Left front),32,6.22,6.52,7.47,78.1%,100.0%,100.0%
6,Brake Caliper -(Left front),20,6.65,2.96,12.69,90.0%,90.0%,95.0%
7,Drive shaft -(Right front),23,8.50,3.60,19.23,82.6%,91.3%,95.7%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Drive shaft -(Right front),23,8.50,3.60,19.23,82.6%,91.3%,95.7%
6,Brake Caliper -(Left front),20,6.65,2.96,12.69,90.0%,90.0%,95.0%
5,Drive shaft -(Left front),32,6.22,6.52,7.47,78.1%,100.0%,100.0%
4,Wheel bearing spindle shaft -(Right rear),20,5.50,0.88,12.90,85.0%,85.0%,100.0%
3,ABS Hydraulic pump -,24,5.39,3.48,7.14,75.0%,100.0%,100.0%
2,Trailing link rear -(Left),20,4.71,2.93,6.28,95.0%,100.0%,100.0%
1,Wheel bearing spindle shaft -(Left rear),21,3.41,1.81,8.75,95.2%,95.2%,100.0%
0,Distributors Vacuum regulator -,20,3.08,2.21,4.28,90.0%,100.0%,100.0%


## 29. Select The Best CatBoost Feature Set

Reuse the tuned CatBoost hyperparameters across every feature-set experiment so the comparison stays focused on feature value instead of model-setup noise.


In [44]:
excluded_features = catboost_excluded_features

feature_sets = {
    "baseline only": features_baseline_only,
    "baseline + traficom_core": features_baseline_plus_traficom,
    "baseline + traficom_core + registry_lifecycle": (
        features_baseline_plus_traficom + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        features_baseline_plus_traficom
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "previous manual all-features set": manual_all_feature_groups,
    "all recommended features": recommended_model_features,
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [45]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


In [46]:
for model_name, feature_list in feature_sets.items():
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    (
        model_current,
        X_train_current_prepared,
        X_validation_current_prepared,
        cat_features_current,
    ) = fit_catboost_model(
        X_train_current,
        y_train_log,
        X_validation_current,
        y_validation_log,
        selected_catboost_params,
    )

    validation_pred_log_current = model_current.predict(X_validation_current_prepared)
    validation_pred_current = np.exp(validation_pred_log_current)
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })


## 30. Validate The Best CatBoost Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [47]:
# Rebuild the best-model validation dataframe if this section is run on its own.
if "best_model_validation_df" not in globals():
    if "best_experiment_name" not in globals():
        if "experiment_results" not in globals() or len(experiment_results) == 0:
            raise NameError(
                "best_experiment_name is not defined. Run the feature-set comparison section first."
            )

        experiment_results_df = pd.DataFrame(experiment_results)
        best_experiment_name = experiment_results_df.sort_values(
            ["validation_MAE", "validation_RMSE"]
        ).iloc[0]["experiment"]

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,231,32.3,36.4,4.09
1,"(5.899, 47.4]",underpredicted,118,41.7,37.3,4.47
2,"(47.4, 82.6]",overpredicted,108,60.9,64.2,3.29
3,"(47.4, 82.6]",underpredicted,227,61.2,56.7,4.52
4,"(82.6, 154.06]",overpredicted,123,113.8,121.8,8.04
5,"(82.6, 154.06]",underpredicted,206,107.6,98.6,9.02
6,"(154.06, 248.42]",overpredicted,153,193.5,202.7,9.20
7,"(154.06, 248.42]",underpredicted,185,214.7,206.4,8.35
8,"(248.42, 4284.0]",overpredicted,152,1009.5,1072.3,62.86
9,"(248.42, 4284.0]",underpredicted,186,781.2,673.0,108.22


In [48]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,4.22,6.26,13.1%,-1.47
1,"(47.4, 82.6]",335,47.6,82.6,61.1,4.12,7.44,6.5%,+0.81
2,"(82.6, 154.06]",329,82.8,153.9,109.9,8.65,18.66,7.7%,+1.48
3,"(154.06, 248.42]",338,154.1,248.3,205.1,8.73,20.47,4.4%,+0.34
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,87.82,335.73,7.2%,+1.40


In [49]:
experiment_results_df = pd.DataFrame(experiment_results)
baseline_row = experiment_results_df.loc[
    experiment_results_df["experiment"] == "baseline only"
].iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

display_columns = [
    "experiment",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

best_experiment_name = experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]


experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})

,experiment,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
5,all recommended features,73,22.6975,-33.9845,1,150.7555,-37.2490,2,0.9293
4,previous manual all-features set,71,22.9711,-33.7109,2,150.3735,-37.6309,1,0.9297
1,baseline + traficom_core,29,55.0203,-1.6616,3,187.7861,-0.2183,3,0.8903
2,baseline + traficom_core + registry_lifecycle,43,55.5975,-1.0844,4,188.4997,+0.4952,6,0.8895
3,baseline + traficom_core + registry_lifecycle + traficom_extended,65,55.9110,-0.7709,5,188.4501,+0.4456,5,0.8895
0,baseline only,16,56.6819,+0.0000,6,188.0045,+0.0000,4,0.8901
